# Binary Image Classification on Synthetic Scenes - Kaggle Competetion Submission
## Team 48 
#### Team Members -
#### Aayush Ranjan || CS25MTECH11002
#### Ankit Kumar Sinha || CS25MTECH11022
#### Mayank Mishra || CS25MTECH11029

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
from PIL import Image, ImageFilter
import pandas as pd
import numpy as np
import cv2


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c),
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(x + self.net(x))


class SEBlock(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(c, c // r), nn.ReLU(inplace=True),
            nn.Linear(c // r, c), nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x).unsqueeze(-1).unsqueeze(-1)


class ShapeNet(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()

        def down_block(ci, co):
            return nn.Sequential(
                ResBlock(ci), SEBlock(ci),
                nn.Conv2d(ci, co, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(co), nn.ReLU(inplace=True),
            )

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.d1 = down_block(32, 64)
        self.d2 = down_block(64, 128)
        self.d3 = down_block(128, 256)
        self.d4 = down_block(256, 512)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.d1(x)
        x = self.d2(x)
        x = self.d3(x)
        x = self.d4(x)
        x = self.pool(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# Edge extraction (material-invariant)
# ---------------------------------------------------------------------------

class EdgeTransform:
    def __call__(self, img):
        arr = np.array(img.convert('L'))
        # Strong bilateral filter to suppress specular highlights on metal objects
        # while preserving true object boundaries
        arr = cv2.bilateralFilter(arr, 9, 150, 150)
        # Canny edges
        canny = cv2.Canny(arr, 30, 100).astype(np.float32) / 255.0
        # Sobel magnitude
        sx = cv2.Sobel(arr, cv2.CV_64F, 1, 0, ksize=3)
        sy = cv2.Sobel(arr, cv2.CV_64F, 0, 1, ksize=3)
        sobel = np.sqrt(sx ** 2 + sy ** 2)
        sobel = np.clip(sobel / sobel.max(), 0, 1).astype(np.float32) if sobel.max() > 0 else sobel.astype(np.float32)
        # Laplacian
        lap = cv2.Laplacian(arr, cv2.CV_64F, ksize=3)
        lap = np.abs(lap)
        lap = np.clip(lap / lap.max(), 0, 1).astype(np.float32) if lap.max() > 0 else lap.astype(np.float32)
        # Stack into 3 channels
        edge_img = np.stack([canny, sobel, lap], axis=2)
        return Image.fromarray((edge_img * 255).astype(np.uint8))


# ---------------------------------------------------------------------------
# Datasets
# ---------------------------------------------------------------------------

class TrainDataset(Dataset):
    def __init__(self, paths, labels, transform, edge_transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.edge_transform = edge_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.edge_transform:
            img = self.edge_transform(img)
        img = self.transform(img)
        return img, self.labels[idx]


class TestDataset(Dataset):
    def __init__(self, paths, transform, edge_transform=None):
        self.paths = paths
        self.transform = transform
        self.edge_transform = edge_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.edge_transform:
            img = self.edge_transform(img)
        img = self.transform(img)
        return img


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _resolve_dir(data_dir, subdir):
    nested = os.path.join(data_dir, subdir, subdir)
    if os.path.isdir(nested):
        return nested
    return os.path.join(data_dir, subdir)


def _load_train_paths(train_dir):
    paths, labels = [], []
    for label in [0, 1]:
        class_dir = os.path.join(train_dir, str(label))
        for p in sorted(glob.glob(os.path.join(class_dir, '*.png'))):
            paths.append(p)
            labels.append(label)
    return paths, labels


def _mixup(imgs, labels, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed_imgs = lam * imgs + (1 - lam) * imgs[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_imgs, mixed_labels


def train_one_model(all_paths, all_labels, device, model_type='rgb', img_size=128,
                    epochs=35, lr=1e-3, patience_limit=8, pos_weight=None,
                    use_mixup=False, batch_size=96, swa_window=None):
    
    is_edge = model_type == 'edge'
    edge_tf = EdgeTransform() if is_edge else None

    # --- Transforms ---
    if is_edge:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.2),
            transforms.RandomGrayscale(p=0.3),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=7, sigma=(0.5, 3.0))], p=0.3),
            transforms.RandomApply([transforms.RandomAutocontrast()], p=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3),
            transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
        ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),
    ])

    # --- Split ---
    n = len(all_paths)
    indices = list(range(n))
    np.random.seed(42)
    np.random.shuffle(indices)
    val_size = int(0.15 * n)
    val_idx, train_idx = indices[:val_size], indices[val_size:]

    train_ds = TrainDataset(
        [all_paths[i] for i in train_idx],
        [all_labels[i] for i in train_idx],
        train_transform, edge_tf)
    val_ds = TrainDataset(
        [all_paths[i] for i in val_idx],
        [all_labels[i] for i in val_idx],
        eval_transform, edge_tf)

    pin = device.type == 'cuda'
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=pin)

    # --- Model ---
    model = ShapeNet(in_channels=3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    if pos_weight is not None:
        pw = torch.tensor([pos_weight], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    else:
        criterion = nn.BCEWithLogitsLoss()

    best_val_acc, best_state, patience = 0.0, None, 0
    swa_model = None
    swa_start = (epochs - swa_window + 1) if swa_window else None

    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.float().to(device)
            if use_mixup:
                imgs, labels = _mixup(imgs, labels, alpha=0.2)
            optimizer.zero_grad()
            criterion(model(imgs).squeeze(1), labels).backward()
            optimizer.step()
        scheduler.step()

        # SWA: average weights over last K epochs
        if swa_start is not None and epoch >= swa_start:
            if swa_model is None:
                swa_model = AveragedModel(model)
            else:
                swa_model.update_parameters(model)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                logits = model(imgs.to(device)).squeeze(1)
                correct += ((logits > 0).long() == labels.to(device).long()).sum().item()
                total += len(labels)
        val_acc = correct / total

        tag = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
            tag = ' ***'
        else:
            patience += 1

        if epoch % 5 == 0 or tag:
            print(f'  [{model_type}] Epoch {epoch:2d}/{epochs}  val={val_acc:.4f}{tag}')

        if patience >= patience_limit:
            print(f'  [{model_type}] Early stop at epoch {epoch} (best={best_val_acc:.4f})')
            break

    # Pick final weights: SWA-averaged if available, else best val checkpoint
    if swa_model is not None:
        # Recompute BN running stats with averaged weights using train data
        update_bn(train_loader, swa_model, device=device)
        # Copy averaged weights back to model
        model.load_state_dict({k.replace('module.', ''): v for k, v in swa_model.state_dict().items()
                               if 'n_averaged' not in k})
        # Verify on val
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                logits = model(imgs.to(device)).squeeze(1)
                correct += ((logits > 0).long() == labels.to(device).long()).sum().item()
                total += len(labels)
        swa_val = correct / total
        print(f'  [{model_type}] SWA val: {swa_val:.4f} (best single epoch: {best_val_acc:.4f})')
    else:
        model.load_state_dict(best_state)
    print(f'  [{model_type}] Best val accuracy: {best_val_acc:.4f}')
    return model, eval_transform, edge_tf


def adapt_bn(model, test_paths, eval_transform, edge_tf, device, batch_size=96):
    ds = TestDataset(test_paths, eval_transform, edge_tf)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=4,
                        pin_memory=(device.type == 'cuda'))
    # Set ONLY BN modules to train mode and reset running stats
    bn_mods = [m for m in model.modules() if isinstance(m, nn.BatchNorm2d)]
    for m in bn_mods:
        m.train()
        m.reset_running_stats()
    # Forward passes to populate running_mean/running_var from test batches
    with torch.no_grad():
        for imgs in loader:
            model(imgs.to(device))
    for m in bn_mods:
        m.eval()


def predict_with_tta(model, test_paths, eval_transform, edge_tf, device, img_size=128, batch_size=96):
    tta_transforms = [
        eval_transform,
        transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=1.0),
            transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3),
        ]),
        transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomVerticalFlip(p=1.0),
            transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3),
        ]),
        transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=1.0), transforms.RandomVerticalFlip(p=1.0),
            transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3),
        ]),
    ]

    model.eval()
    all_logits = np.zeros(len(test_paths))

    for tf in tta_transforms:
        ds = TestDataset(test_paths, tf, edge_tf)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4,
                            pin_memory=(device.type == 'cuda'))
        idx = 0
        with torch.no_grad():
            for imgs in loader:
                logits = model(imgs.to(device)).squeeze(1).cpu().numpy()
                all_logits[idx:idx + len(logits)] += logits
                idx += len(logits)

    all_logits /= len(tta_transforms)
    return all_logits


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def _select_pseudo_labels(rgb_logits, edge_logits, test_paths, neg_thresh=0.15):
    rgb_sig = 1.0 / (1.0 + np.exp(-rgb_logits))
    edge_sig = 1.0 / (1.0 + np.exp(-edge_logits))
    both_neg = (rgb_sig < neg_thresh) & (edge_sig < neg_thresh)
    neg_idx = np.where(both_neg)[0]
    print(f'  Confident negatives (rgb<{neg_thresh} & edge<{neg_thresh}): {len(neg_idx)}')

    paths = [test_paths[i] for i in neg_idx]
    labels = [0] * len(neg_idx)
    return paths, labels


def generate_predictions(data_dir):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    train_dir = _resolve_dir(data_dir, 'train')
    all_paths, all_labels = _load_train_paths(train_dir)
    print(f'Train: {len(all_paths)} images ({all_labels.count(0)} neg, {all_labels.count(1)} pos)')

    test_dir = _resolve_dir(data_dir, 'test')
    test_paths = sorted(glob.glob(os.path.join(test_dir, '*.png')))
    print(f'Test: {len(test_paths)} images')

    IMG_SIZE = 192          
    BATCH_SIZE = 96         
    POS_WEIGHT = 0.6        
    RGB_EPOCHS = 60         
    EDGE_EPOCHS = 40        
    SWA_RGB = 15            
    SWA_EDGE = 10

    common_kw = dict(img_size=IMG_SIZE, lr=1e-3, batch_size=BATCH_SIZE,
                     pos_weight=POS_WEIGHT)

    # --- Round 1: initial training on labeled data only ---
    print('\n=== Round 1: RGB model ===')
    rgb_model, rgb_eval_tf, _ = train_one_model(
        all_paths, all_labels, device, model_type='rgb',
        epochs=RGB_EPOCHS, patience_limit=RGB_EPOCHS,  # disable early stop (let SWA collect)
        use_mixup=True, swa_window=SWA_RGB, **common_kw)

    print('\n=== Round 1: Edge model ===')
    edge_model, edge_eval_tf, edge_tf = train_one_model(
        all_paths, all_labels, device, model_type='edge',
        epochs=EDGE_EPOCHS, patience_limit=EDGE_EPOCHS,
        use_mixup=False, swa_window=SWA_EDGE, **common_kw)

    # --- Pseudo-labeling: only confident negatives ---
    print('\n=== Generating pseudo-labels (negatives only) ===')
    # BN-adapt before pseudo-label pass too — better calibrated confidences
    adapt_bn(rgb_model, test_paths, rgb_eval_tf, None, device, BATCH_SIZE)
    adapt_bn(edge_model, test_paths, edge_eval_tf, edge_tf, device, BATCH_SIZE)
    rgb_logits_r1 = predict_with_tta(rgb_model, test_paths, rgb_eval_tf, None, device, IMG_SIZE, BATCH_SIZE)
    edge_logits_r1 = predict_with_tta(edge_model, test_paths, edge_eval_tf, edge_tf, device, IMG_SIZE, BATCH_SIZE)
    pseudo_paths, pseudo_labels = _select_pseudo_labels(
        rgb_logits_r1, edge_logits_r1, test_paths, neg_thresh=0.15)

    del rgb_model, edge_model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    combined_paths = all_paths + pseudo_paths
    combined_labels = all_labels + pseudo_labels
    n_pos = sum(combined_labels)
    n_neg = len(combined_labels) - n_pos
    print(f'  Combined train set: {len(combined_paths)} images ({n_pos} pos, {n_neg} neg)')

    # --- Round 2: retrain on labeled + pseudo-labeled data ---
    print('\n=== Round 2: RGB model (with neg pseudo-labels) ===')
    rgb_model, rgb_eval_tf, _ = train_one_model(
        combined_paths, combined_labels, device, model_type='rgb',
        epochs=RGB_EPOCHS, patience_limit=RGB_EPOCHS,
        use_mixup=True, swa_window=SWA_RGB, **common_kw)

    print('\n=== Round 2: Edge model (with neg pseudo-labels) ===')
    edge_model, edge_eval_tf, edge_tf = train_one_model(
        combined_paths, combined_labels, device, model_type='edge',
        epochs=EDGE_EPOCHS, patience_limit=EDGE_EPOCHS,
        use_mixup=False, swa_window=SWA_EDGE, **common_kw)

    # --- BN-adapt to test distribution before final inference ---
    print('\n=== BN test-time adaptation ===')
    adapt_bn(rgb_model, test_paths, rgb_eval_tf, None, device, BATCH_SIZE)
    adapt_bn(edge_model, test_paths, edge_eval_tf, edge_tf, device, BATCH_SIZE)

    # --- Final inference with TTA ---
    print('\n=== Final inference ===')
    rgb_logits = predict_with_tta(rgb_model, test_paths, rgb_eval_tf, None, device, IMG_SIZE, BATCH_SIZE)
    edge_logits = predict_with_tta(edge_model, test_paths, edge_eval_tf, edge_tf, device, IMG_SIZE, BATCH_SIZE)

    ensemble_logits = 0.5 * rgb_logits + 0.5 * edge_logits
    preds = (ensemble_logits > 0).astype(int)

    ids = [os.path.basename(p) for p in test_paths]
    df = pd.DataFrame({'ID': ids, 'Label': preds})
    df.to_csv('submission.csv', index=False)
    print(f'Saved submission.csv  ({preds.sum()} pos / {len(preds)} total)')
    print(f'RGB predicts:  {(rgb_logits > 0).sum()} pos')
    print(f'Edge predicts: {(edge_logits > 0).sum()} pos')


if __name__ == '__main__':
    generate_predictions('/kaggle/input/competitions/iith-deep-learning-2026-hackathon')